# NB-05-01 — Uncertainty Workbench

Bayes' theorem, exact inference in the alarm Bayesian network, naive Bayes and a fuzzy controller.
No external data, fixed seeds, every printed number reproducible.

The network section uses `code/alarm_network.py`, which stores **complete** conditional probability
tables and asserts that its joint distribution sums to 1.

In [1]:
import math, pathlib, sys, random
random.seed(0)

CODE = None
for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (base / "code" / "alarm_network.py").exists():
        CODE = base / "code"
        break
if CODE is None:
    CODE = pathlib.Path.cwd().parent / "code"
sys.path.insert(0, str(CODE))
import alarm_network as net
print("code on path:", CODE)

code on path: C:\Users\Nadeem\Documents\Artificial-Intelligence-Course\modules\M05-uncertainty-bayesian-reasoning-and-fuzzy-logic\code


## 1. Bayes' theorem and the base rate

Same test accuracy, three prevalences.

In [2]:
def bayes(prior, sens, spec):
    return prior * sens / (prior * sens + (1 - prior) * (1 - spec))

for prior in (0.30, 0.02, 0.01):
    print(f"prior={prior:4.2f} -> posterior={bayes(prior, 0.99, 0.99):.3f}")

prior=0.30 -> posterior=0.977
prior=0.02 -> posterior=0.669
prior=0.01 -> posterior=0.500


## 2. The network must sum to 1

This is the check the defective version failed: its joint totalled 0.005 because the alarm table
stored only $P(A=T\mid b,e)$ and reused that value for the $a=F$ branch.

In [3]:
table = net.joint_table()
print("joint entries:", len(table))
print("joint total:  ", round(sum(table.values()), 12))
print("P(B=True):    ", round(sum(v for k, v in table.items() if k[0]), 10), " (prior 0.001)")
print("P(E=True):    ", round(sum(v for k, v in table.items() if k[1]), 10), " (prior 0.002)")
p_b_and_e = sum(v for k, v in table.items() if k[0] and k[1])
print("P(B,E):       ", f"{p_b_and_e:.3e}", " P(B)P(E):", f"{0.001 * 0.002:.3e}", "-> independent")

joint entries: 32
joint total:   1.0
P(B=True):     0.001  (prior 0.001)
P(E=True):     0.002  (prior 0.002)
P(B,E):        2.000e-06  P(B)P(E): 2.000e-06 -> independent


## 3. Step-by-step enumeration for $P(B \mid j, m)$

Two branches, each summing over the hidden E and A. These values match the hand derivation in
L05-02 section 4.3.

In [4]:
def branch(b, j, m):
    return sum(net.joint(b, e, a, j, m) for e in (True, False) for a in (True, False))

b_true, b_false = branch(True, True, True), branch(False, True, True)
print(f"P(B=T, j, m) = {b_true:.8f}")
print(f"P(B=F, j, m) = {b_false:.8f}")
print(f"P(j, m)      = {b_true + b_false:.8f}")
print(f"posterior    = {b_true / (b_true + b_false):.4f}   (published: 0.284)")

P(B=T, j, m) = 0.00059224
P(B=F, j, m) = 0.00149186
P(j, m)      = 0.00208410
posterior    = 0.2842   (published: 0.284)


## 4. Evidence comparison

Each row is a separate exact-inference query. The posterior must move with the evidence, in the
direction the graph predicts.

In [5]:
print(f"{'evidence':26s} {'P(B|e)':>8s} {'P(e)':>11s}")
for label, ev, posterior, p_ev in net.evidence_table():
    print(f"{label:26s} {posterior:8.4f} {p_ev:11.7f}")

evidence                     P(B|e)        P(e)
no evidence                  0.0010   1.0000000
John calls                   0.0163   0.0521390
Mary calls                   0.0561   0.0117363
both call                    0.2842   0.0020841
John only                    0.0051   0.0500549
Mary only                    0.0069   0.0096522
neither calls                0.0001   0.9382088
earthquake                   0.0010   0.0020000
earthquake, both call        0.0033   0.0003669


## 5. Explaining away

Earthquake evidence alone leaves burglary at its prior, because B and E are independent. Add the
alarm's effects and the same earthquake evidence *defeats* them.

In [6]:
p_prior = net.enumerate_posterior("B", {})[0]
p_calls = net.enumerate_posterior("B", {"J": True, "M": True})[0]
p_quake = net.enumerate_posterior("B", {"E": True})[0]
p_both = net.enumerate_posterior("B", {"E": True, "J": True, "M": True})[0]
print(f"P(B)           = {p_prior:.4f}")
print(f"P(B | j, m)    = {p_calls:.4f}")
print(f"P(B | e)       = {p_quake:.4f}   (unchanged: B independent of E)")
print(f"P(B | e, j, m) = {p_both:.4f}   (explaining away: {p_calls / p_both:.0f}x lower)")
print(f"P(A | j)       = {net.enumerate_posterior('A', {'J': True})[0]:.4f}")

P(B)           = 0.0010
P(B | j, m)    = 0.2842
P(B | e)       = 0.0010   (unchanged: B independent of E)
P(B | e, j, m) = 0.0033   (explaining away: 87x lower)
P(A | j)       = 0.0434


## 6. Tests

Normalisation, evidence handling, published posteriors, evidence combinations, and rejection of
invalid input. `run_tests()` raises `AssertionError` if any check fails.

In [7]:
print("module self-tests passed:", net.run_tests())

checks = {
    "normalisation": abs(sum(net.joint_table().values()) - 1.0) < 1e-12,
    "published posterior 0.284": abs(net.enumerate_posterior("B", {"J": True, "M": True})[0] - 0.28417) < 1e-4,
    "independence B,E": abs(net.enumerate_posterior("B", {"E": True})[0] - 0.001) < 1e-12,
    "evidence monotone": (net.enumerate_posterior("B", {})[0]
                          < net.enumerate_posterior("B", {"J": True})[0]
                          < net.enumerate_posterior("B", {"J": True, "M": True})[0]),
    "explaining away": (net.enumerate_posterior("B", {"E": True, "J": True, "M": True})[0]
                        < net.enumerate_posterior("B", {"J": True, "M": True})[0]),
}
for name, ok in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

for bad in ({"X": True}, {"J": 1}, {"J": "yes"}):
    try:
        net.enumerate_posterior("B", bad)
    except ValueError as exc:
        print("  rejected", bad, "->", exc)
    else:
        print("  FAIL: accepted", bad)

module self-tests passed: True
  PASS  normalisation
  PASS  published posterior 0.284
  PASS  independence B,E
  PASS  evidence monotone
  PASS  explaining away
  rejected {'X': True} -> unknown node 'X'; expected one of ('B', 'E', 'A', 'J', 'M')
  rejected {'J': 1} -> evidence for 'J' must be True or False, got 1
  rejected {'J': 'yes'} -> evidence for 'J' must be True or False, got 'yes'


## 7. Naive Bayes on a tiny corpus

Smoothing keeps an unseen word from vetoing a class.

In [8]:
from collections import Counter, defaultdict


class NB:
    def __init__(self, alpha=1.0):
        self.alpha, self.c, self.n, self.v = alpha, defaultdict(Counter), Counter(), set()

    def fit(self, docs, labels):
        for d, y in zip(docs, labels):
            self.n[y] += 1
            for w in d.split():
                self.c[y][w] += 1
                self.v.add(w)
        self.tot = {y: sum(self.c[y].values()) for y in self.n}
        self.prior_den = sum(self.n.values())

    def predict(self, doc):
        s = {}
        for y in self.n:
            p = math.log(self.n[y] / self.prior_den)
            for w in doc.split():
                p += math.log((self.c[y][w] + self.alpha) / (self.tot[y] + self.alpha * len(self.v)))
            s[y] = p
        return max(s, key=s.get), s


docs = ["free money now", "free invoice attached", "meeting at noon",
        "free meeting tomorrow", "invoice for meeting room"]
labels = ["spam", "spam", "ham", "ham", "ham"]
m = NB()
m.fit(docs, labels)
for q in ("free meeting", "invoice attached", "unknownword"):
    print(q, "->", m.predict(q)[0])

free meeting -> ham
invoice attached -> spam
unknownword -> ham


## 8. Membership functions and a Sugeno controller

Cold / warm / hot over 0-40 C, then a ventilation rule base.

In [9]:
def tri(x, a, b, c):
    if x <= a or x >= c:
        return 0.0
    return (x - a) / (b - a) if x <= b else (c - x) / (c - b)


cold = lambda x: tri(x, 0, 0, 18)
warm = lambda x: tri(x, 10, 20, 30)
hot = lambda x: tri(x, 22, 40, 40)
for t in (8, 18, 24, 33):
    print(t, round(cold(t), 2), round(warm(t), 2), round(hot(t), 2))
print("warm AND hot:", min(warm(24), hot(24)), "| warm OR hot:", max(warm(24), hot(24)))


def fan(temp, hum):
    hot_t, warm_t, cold_t = hot(temp), warm(temp), cold(temp)
    wet = tri(hum, 40, 100, 100)
    rules = [(hot_t, "high"), (min(warm_t, wet), "high"), (warm_t, "medium"), (cold_t, "low")]
    speeds = {"low": 20.0, "medium": 50.0, "high": 90.0}
    num = sum(w * speeds[s] for w, s in rules)
    den = sum(w for w, _ in rules)
    return (num / den if den else 0.0), round(den, 3)


for t, h in ((26, 70), (5, 20), (35, 95), (20, 50)):
    print(t, h, "->", round(fan(t, h)[0], 1), "% damper, strength", fan(t, h)[1])

8 0.56 0.0 0.0
18 0.0 0.8 0.0
24 0.0 0.6 0.11
33 0.0 0.0 0.61
warm AND hot: 0.1111111111111111 | warm OR hot: 0.6
26 70 -> 74.3 % damper, strength 1.022
5 20 -> 20.0 % damper, strength 0.722
35 95 -> 90.0 % damper, strength 0.722
20 50 -> 55.7 % damper, strength 1.167


## What you should take away

- A network's joint must sum to 1. The defective version totalled 0.005, so every posterior was wrong
  while still looking plausible.
- Enumeration is the factorisation summed over hidden variables; each branch is checkable by hand.
- Evidence moves the posterior monotonically: 0.001 -> 0.016 (John) -> 0.284 (both).
- Explaining away is large here: adding the earthquake cuts 0.284 to 0.0033.
- Independence shows up as an unchanged posterior: $P(B\mid E) = P(B)$.

Next: [LAB-05-01](../labs/lab-05-01-bayesian-and-fuzzy-decision-system.md).